# FL + LoRA + CLIP on RSTPReid (non-IID by camera)

Minimal baseline for the thesis on PEFT + Federated Learning + TBPS.

**Architecture:** CLIP ViT-B/16 dual-encoder + SDM loss (IRRA's IRR/MLM/ID heads dropped)
**FL:** manual FedAvg, 15 clients = 15 cameras, only LoRA parameters are transmitted
**Reference points:** CLIP baseline on RSTPReid = **54.05 R@1** (IRRA repo), full IRRA = 60.20

> Runtime -> Change runtime type -> **GPU** (A100 or L4 if available; T4 works too but ~3x slower)

## 1. Check GPU

In [ ]:
!nvidia-smi
import torch; print('torch', torch.__version__, '| cuda', torch.cuda.is_available())

## 2. Install dependencies

In [ ]:
# Pinned to the exact version this pipeline was developed/tested against.
# "transformers>=4.40" is NOT enough: Colab's preinstalled version already
# satisfies ">=4.40", so pip skips reinstalling, and a different CLIPModel
# implementation can make get_image_features()/get_text_features() return
# something else (e.g. the raw BaseModelOutputWithPooling instead of the
# projected tensor) -> AttributeError deep inside training.
!pip -q install "transformers==4.57.6" pillow
import transformers; print('transformers', transformers.__version__)

## 3. Mount Drive and point to the dataset

Expected folder structure:
```
RSTPReid/
  imgs/
    0001_c1_0001.jpg
    ...
  data_captions.json
```

**Speed tip:** reading ~20k small images directly from Google Drive is very slow. Zip it on Drive, then unzip into `/content` (Colab's local disk). The cell below does that.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Edit this path to match your Drive layout
ZIP_ON_DRIVE = '/content/drive/MyDrive/datasets/RSTPReid.zip'

import os, zipfile, time
DATA_ROOT = '/content/RSTPReid'
if not os.path.exists(DATA_ROOT):
    t = time.time()
    # NOTE: this zip has no top-level 'RSTPReid/' folder inside it (imgs/ and
    # data_captions.json sit at the zip root) -> extract DIRECTLY into
    # DATA_ROOT, not into /content.
    with zipfile.ZipFile(ZIP_ON_DRIVE) as z:
        z.extractall(DATA_ROOT)
    print(f'extracted in {time.time()-t:.0f}s')

print(sorted(os.listdir(DATA_ROOT))[:5])
print('num images:', len(os.listdir(os.path.join(DATA_ROOT, 'imgs'))))

## 4. Get the code

In [ ]:
!git clone https://github.com/thanhtungdo2211/master-thesis.git /content/master-thesis

%cd /content/master-thesis/experiments
!ls clip_lora_fedavg/

# Alternative, if you'd rather sync via Drive instead of git:
# !cp -r /content/drive/MyDrive/experiments /content/experiments
# %cd /content/experiments

## 5. Smoke test (~5 minutes)

Runs 2 rounds, 3 steps per client, 3 clients. Purpose: catch shape/path errors before the real run.

In [ ]:
!python -m clip_lora_fedavg.main \
  --root /content/RSTPReid \
  --out_dir /content/runs/smoke \
  --partition camera \
  --rounds 2 --eval_every 2 \
  --max_clients_per_round 3 --debug_steps 3 \
  --batch_size 16 --num_workers 2

## 6. Real runs

Run these 4 configs in order. Prioritize B5 and B3 first — this pair gives you the non-IID gap.

| ID | partition | tuning | Role |
|---|---|---|---|
| B5 | camera | lora r=4 | **Main focus** |
| B3 | iid | lora r=4 | Control, isolates the non-IID effect |
| B4 | camera | full | Measures the cost of PEFT |
| B2 | iid | full | |

Rough time estimate on an A100: ~580 steps/round at batch 64 -> ~1.5-2 min/round, 50 rounds ~= 1.5 hours. On a T4, multiply by ~3.

In [ ]:
# B5: FL camera non-IID + LoRA r=4  <-- run this one first
!python -m clip_lora_fedavg.main \
  --root /content/RSTPReid \
  --out_dir /content/drive/MyDrive/runs/B5_camera_lora4 \
  --partition camera \
  --tuning lora --lora_rank 4 --lora_alpha 8 \
  --rounds 50 --local_epochs 1 --eval_every 5 \
  --batch_size 64 --lr 1e-4

In [ ]:
# B3: FL IID + LoRA r=4
!python -m clip_lora_fedavg.main \
  --root /content/RSTPReid \
  --out_dir /content/drive/MyDrive/runs/B3_iid_lora4 \
  --partition iid --num_clients 15 \
  --tuning lora --lora_rank 4 --lora_alpha 8 \
  --rounds 50 --local_epochs 1 --eval_every 5 \
  --batch_size 64 --lr 1e-4

In [ ]:
# B4: FL camera non-IID + full fine-tuning (10x lower lr)
!python -m clip_lora_fedavg.main \
  --root /content/RSTPReid \
  --out_dir /content/drive/MyDrive/runs/B4_camera_full \
  --partition camera \
  --tuning full \
  --rounds 50 --local_epochs 1 --eval_every 5 \
  --batch_size 32 --lr 1e-5

## 6b. Quick progress check (while a run is still training)

`log.csv` is appended after every round, and `checkpoint.pt` is updated
every `--ckpt_every` rounds — both are readable while training is still
running (e.g. in a second Colab cell, or from another notebook tab). This
is also how you resume: if a run gets cut off, rerunning the exact same
`main` command from section 6 auto-continues from `checkpoint.pt` instead
of starting over from round 1 (unless you pass `--fresh`).

In [ ]:
import json, os
import pandas as pd
import matplotlib.pyplot as plt
import torch

RUN_DIR = '/content/drive/MyDrive/runs/B5_camera_lora4'  # point this at the run you want to check

log_path = os.path.join(RUN_DIR, 'log.csv')
ckpt_path = os.path.join(RUN_DIR, 'checkpoint.pt')

if os.path.exists(ckpt_path):
    ckpt = torch.load(ckpt_path, map_location='cpu', weights_only=False)
    print(f"checkpoint: round {ckpt['round']} | best R@1 so far {ckpt['best_r1']:.2f} | "
          f"uplink so far {ckpt['cum_uplink']/1024:.2f} GB | elapsed {ckpt['elapsed_s']/60:.1f} min")
else:
    print('no checkpoint.pt yet -- training may not have reached --ckpt_every rounds')

if os.path.exists(log_path):
    df = pd.read_csv(log_path)
    ev = df[df['R@1'].notna()]
    print(f"\n{len(df)} rounds logged, {len(ev)} of them evaluated")
    display(ev.tail(5))
    if len(ev) >= 2:
        fig, ax = plt.subplots(figsize=(6, 4))
        ax.plot(ev['round'], ev['R@1'], marker='o')
        ax.axhline(54.05, ls='--', c='gray')
        ax.text(ev['round'].iloc[0], 54.6, 'CLIP baseline 54.05', fontsize=8, c='gray')
        ax.set_xlabel('round'); ax.set_ylabel('R@1 (%)'); ax.set_title(os.path.basename(RUN_DIR))
        ax.grid(alpha=.3)
        plt.show()
else:
    print('no log.csv yet -- training may still be on round 1')

## 7. Plot curves and compare

In [ ]:
import pandas as pd, matplotlib.pyplot as plt

RUNS = {
    'B5 camera + LoRA r=4': '/content/drive/MyDrive/runs/B5_camera_lora4/log.csv',
    'B3 IID + LoRA r=4':    '/content/drive/MyDrive/runs/B3_iid_lora4/log.csv',
}

fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
for name, path in RUNS.items():
    try:
        df = pd.read_csv(path)
    except FileNotFoundError:
        print(f'not found yet: {path}'); continue
    ev = df[df['R@1'].notna()]
    ax[0].plot(ev['round'], ev['R@1'], marker='o', label=name)
    ax[1].plot(ev['cum_uplink_MB']/1024, ev['R@1'], marker='o', label=name)

ax[0].axhline(54.05, ls='--', c='gray'); ax[0].text(1, 54.6, 'CLIP baseline 54.05 (IRRA repo)', fontsize=8, c='gray')
ax[0].axhline(60.20, ls='--', c='crimson'); ax[0].text(1, 60.8, 'IRRA full 60.20', fontsize=8, c='crimson')
ax[0].set_xlabel('round'); ax[0].set_ylabel('R@1 (%)'); ax[0].set_title('Convergence over rounds'); ax[0].legend(); ax[0].grid(alpha=.3)
ax[1].set_xlabel('cumulative uplink (GB)'); ax[1].set_ylabel('R@1 (%)'); ax[1].set_title('Accuracy vs communication cost'); ax[1].legend(); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.show()

# Summary table
rows = []
import json, os
for name, path in RUNS.items():
    d = os.path.dirname(path)
    if not os.path.exists(path): continue
    df = pd.read_csv(path); ev = df[df['R@1'].notna()]
    cfg = json.load(open(os.path.join(d, 'args.json')))
    best = ev.loc[ev['R@1'].idxmax()]
    rows.append({'run': name, 'best R@1': best['R@1'], 'R@5': best['R@5'], 'R@10': best['R@10'],
                 'mAP': best['mAP'], 'mINP': best['mINP'],
                 'trainable': cfg['trainable_params'],
                 'MB/round/client': round(cfg['uplink_mb_fp32'], 2),
                 'total uplink GB': round(best['cum_uplink_MB']/1024, 2)})
pd.DataFrame(rows)